# Flow-Aware Temporal Pattern Mining for Multi-Stage Network Intrusion Detection (NIDS)

**Dataset:** CIC-IDS2017 &middot; **Split:** strictly chronological (Train = Days 1-2, Val = Day 3, Test = Days 4-7)

A 13-stage, session-aware pipeline: three parallel detectors (Random Forest, BiLSTM, XGBoost)
fused with frequent/sequential pattern mining and a temporal attack-state graph, turned into a
calibrated risk decision, evaluated under streaming conditions, and explained per-alert with a
three-modality evidence chain.

No random shuffling occurs anywhere in this pipeline. All 15 CIC-IDS2017 class labels are
retained throughout — rare classes (Heartbleed, Infiltration) are handled via class weighting,
never dropped or merged.

## Setup

Resolves the project root so `config` and `nids` import correctly regardless of the
directory Jupyter was launched from, then loads every pipeline module once.

In [ ]:
import os
import sys
import pathlib
import logging
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

PROJECT_ROOT = pathlib.Path.cwd()
if not (PROJECT_ROOT / "config.py").exists():
    for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
        if (parent / "config.py").exists() and (parent / "src" / "nids").exists():
            PROJECT_ROOT = parent
            break
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config
from nids import (
    data_loading, preprocessing, feature_engineering, sessions, events,
    models, fusion, pattern_mining, attack_graph, risk, streaming, explainability,
)
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score

FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
CATEGORICAL_PALETTE = sns.color_palette(config.PALETTE_NAME, n_colors=max(config.K, 10))

print(f"Project root: {PROJECT_ROOT}")

## [STAGE 1] Data Loading & Chronological Split

Loads all 7 CIC-IDS2017 day CSVs, fixing the dataset's known data-quality issues
(leading-space column names, embedded duplicate header rows, inf values, unparsable
timestamps) inside `data_loading.load_day`, then builds the chronological split.
`NIDS_DATASET_DIR` can override `config.DATASET_DIR` without editing the file.

In [ ]:
DATASET_DIR = pathlib.Path(os.environ["NIDS_DATASET_DIR"]) if os.environ.get("NIDS_DATASET_DIR") else config.DATASET_DIR
print(f"Loading CIC-IDS2017 from: {DATASET_DIR}")

df_train, df_val, df_test = data_loading.load_all_days(DATASET_DIR)

print(f"Train: {df_train.shape} | Val: {df_val.shape} | Test: {df_test.shape}")
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    print(f"\n{name} class distribution:")
    print(split["Label"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
combined_counts = pd.concat(
    [
        df_train["Label"].value_counts().rename("Train"),
        df_val["Label"].value_counts().rename("Val"),
        df_test["Label"].value_counts().rename("Test"),
    ],
    axis=1,
).fillna(0)
combined_counts.plot(kind="bar", ax=ax, logy=True, color=CATEGORICAL_PALETTE[:3])
ax.set_ylabel("Flow count (log scale)")
ax.set_xlabel("Class")
ax.set_title("Stage 1: Class Distribution by Split (log scale)")
ax.legend(title="Split")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage1_class_dist.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 2] Preprocessing & Class Imbalance Handling

Produces two parallel branches: **Branch A** (original data, feeds the BiLSTM,
pattern mining, and the attack graph) and **Branch B** (SMOTE+ENN augmented,
feeds RF/XGBoost only — the "temporal-stream firewall"). Classes below
`SMOTE_MIN_SAMPLES` (Heartbleed, Infiltration) are left to class weighting.

In [ ]:
train_clean, val_clean, test_clean, scaler, feat_cols = preprocessing.preprocess(df_train, df_val, df_test)
le, CLASSES, K = preprocessing.encode_labels(train_clean, val_clean, test_clean)
class_weights = preprocessing.compute_class_weights(train_clean, le, K)
print(f"K={K} classes={CLASSES}")
print("Class weights:", class_weights)

X_train = train_clean[feat_cols].to_numpy(dtype=np.float64)
y_train = train_clean["LabelID"].to_numpy()
X_val = val_clean[feat_cols].to_numpy(dtype=np.float64)
y_val = val_clean["LabelID"].to_numpy()
X_test = test_clean[feat_cols].to_numpy(dtype=np.float64)
y_test = test_clean["LabelID"].to_numpy()

class_counts = {int(c): int(n) for c, n in zip(*np.unique(y_train, return_counts=True))}
X_train_B, y_train_B = preprocessing.smote_enn_augment(X_train, y_train, class_counts, K, seed=config.SEED)

print(f"\nBranch A (original):     X={X_train.shape}, y={y_train.shape}")
print(f"Branch B (SMOTE+ENN):    X={X_train_B.shape}, y={y_train_B.shape}")

In [ ]:
before_counts = pd.Series(y_train).value_counts().sort_index()
after_counts = pd.Series(y_train_B).value_counts().sort_index()
class_names = [le.inverse_transform([i])[0] for i in before_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(before_counts))
width = 0.35
axes[0].bar(x - width / 2, before_counts.values, width, label="Before SMOTE+ENN", color=CATEGORICAL_PALETTE[0])
axes[0].bar(
    x + width / 2, [after_counts.get(i, 0) for i in before_counts.index], width,
    label="After SMOTE+ENN", color=CATEGORICAL_PALETTE[1],
)
axes[0].set_yscale("log")
axes[0].set_xticks(x)
axes[0].set_xticklabels(class_names, rotation=90)
axes[0].set_ylabel("Sample count (log scale)")
axes[0].set_title("Class Counts Before vs After SMOTE+ENN")
axes[0].legend()

weight_vals = [class_weights[i] for i in before_counts.index]
axes[1].bar(x, weight_vals, color=CATEGORICAL_PALETTE[2])
axes[1].set_xticks(x)
axes[1].set_xticklabels(class_names, rotation=90)
axes[1].set_ylabel("Class weight $W_c$")
axes[1].set_title("Per-Class Weights ($W_c = N_{train} / (K \\cdot N_c)$)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage2_imbalance.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 3] Feature Engineering & Group Assignment

Splits `feat_cols` into four semantic groups by domain knowledge, with a
mutual-information fallback if this particular CSV variant's column names
don't fill a group to its target size. `Source Port`/`Source IP`/`Destination IP`/
`Flow ID` are dropped unconditionally (host memorisation); `Destination Port`
is kept (protocol semantics).

In [ ]:
group_A, group_B, group_C, group_D = feature_engineering.assign_feature_groups(feat_cols, X_train=X_train, y_train=y_train)

print(f"Group A ({len(group_A)}/{config.GROUP_A_SIZE}) flow-statistical -> RF:\n  {group_A}\n")
print(f"Group B ({len(group_B)}/{config.GROUP_B_SIZE}) protocol/communication -> shared RF+XGB:\n  {group_B}\n")
print(f"Group C ({len(group_C)}/{config.GROUP_C_SIZE}) derived temporal -> LSTM:\n  {group_C}\n")
print(f"Group D ({len(group_D)}/{config.GROUP_D_SIZE}) TCP behavioral flags -> XGBoost:\n  {group_D}\n")

assert len(group_A) == config.GROUP_A_SIZE, "Group A size mismatch"
assert len(group_B) == config.GROUP_B_SIZE, "Group B size mismatch"
assert len(group_C) == config.GROUP_C_SIZE, "Group C size mismatch"
assert len(group_D) == config.GROUP_D_SIZE, "Group D size mismatch"

feat_col_index = {c: i for i, c in enumerate(feat_cols)}
ab_cols = group_A + group_B
bd_cols = group_B + group_D
ab_idx = [feat_col_index[c] for c in ab_cols]
bd_idx = [feat_col_index[c] for c in bd_cols]

## [STAGE 4] Bidirectional Session Reconstruction

Groups flows into sessions per split via a symmetric 5-tuple key (60s timeout,
FIN/RST early termination, day-boundary isolation, 3600s hard cap). Called once
per split so no session can span the train/test boundary.

In [ ]:
sessions_df_train = sessions.reconstruct_sessions(train_clean, session_prefix="TR")
sessions_df_val = sessions.reconstruct_sessions(val_clean, session_prefix="VAL")
sessions_df_test = sessions.reconstruct_sessions(test_clean, session_prefix="TE")

for name, sdf in [("Train", sessions_df_train), ("Val", sessions_df_val), ("Test", sessions_df_test)]:
    lengths = sdf.groupby("SessionID").size()
    print(
        f"{name}: {sdf['SessionID'].nunique()} sessions from {len(sdf)} flows "
        f"(mean length={lengths.mean():.2f}, median={lengths.median():.1f}, "
        f"single-event={int((lengths == 1).sum())})"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, sdf) in enumerate([("Train", sessions_df_train), ("Val", sessions_df_val), ("Test", sessions_df_test)]):
    lengths = sdf.groupby("SessionID").size()
    ax.hist(lengths, bins=30, alpha=0.6, label=name, color=CATEGORICAL_PALETTE[i])
ax.set_xlabel("Session length (# flows)")
ax.set_ylabel("Frequency")
ax.set_title("Stage 4: Session Length Distribution")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage4_sessions.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 5] Behavioral Event Encoding

Applies the 10-token rule-based vocabulary (`events.assign_token`, never
reads `Label`) to every flow, then pads/truncates each session to
`MAX_SEQ_LEN`. Also defines `build_session_features`, the glue used from
Stage 6 onward: since RF/XGBoost are flow-level classifiers but fusion is
session-level, a session's feature vector is the mean of its flows'
(already-scaled) feature values.

In [ ]:
train_sessions = events.encode_sessions(sessions_df_train, max_seq_len=config.MAX_SEQ_LEN)
val_sessions = events.encode_sessions(sessions_df_val, max_seq_len=config.MAX_SEQ_LEN)
test_sessions = events.encode_sessions(sessions_df_test, max_seq_len=config.MAX_SEQ_LEN)

token_counts = Counter()
for rec in train_sessions:
    token_counts.update(t for t in rec["token_seq"] if t is not None)
print("Training token distribution:", dict(token_counts))


def build_session_features(sessions_df, session_records, feat_cols, ab_idx, bd_idx):
    """Session-level feature vector = mean-pooled flow features across the session."""
    means = sessions_df.groupby("SessionID")[feat_cols].mean()
    ordered = means.reindex([r["session_id"] for r in session_records]).to_numpy(dtype=np.float64)
    return {
        "X_AB": ordered[:, ab_idx],
        "X_BD": ordered[:, bd_idx],
        "token_ids": np.array([r["token_ids"] for r in session_records], dtype=np.int32),
    }


safe_label_transform = preprocessing.safe_label_transform

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
tokens_sorted = sorted(token_counts, key=lambda t: -token_counts[t])
ax.bar(tokens_sorted, [token_counts[t] for t in tokens_sorted], color=CATEGORICAL_PALETTE[3])
ax.set_ylabel("Frequency")
ax.set_title("Stage 5: Behavioral Token Frequency (Training Sessions)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage5_tokens.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 6] Multi-Model Parallel Detection (RF + BiLSTM + XGBoost)

RF and XGBoost train on flow-level Branch B (SMOTE+ENN) data restricted to
their respective feature groups; the BiLSTM trains only on original
(un-augmented) session token sequences — the temporal-stream firewall.

In [ ]:
X_train_B_AB = X_train_B[:, ab_idx]
X_val_AB = X_val[:, ab_idx]
rf, rf_val_metrics = models.train_rf(X_train_B_AB, y_train_B, X_val_AB, y_val, class_weights, ab_cols)
print(f"RF val macro-F1: {rf_val_metrics['macro_f1']:.4f}")

X_train_B_BD = X_train_B[:, bd_idx]
X_val_BD = X_val[:, bd_idx]
xgb, xgb_val_metrics = models.train_xgb(X_train_B_BD, y_train_B, X_val_BD, y_val, class_weights, bd_cols)
print(f"XGBoost val macro-F1: {xgb_val_metrics['macro_f1']:.4f}")

lstm, lstm_val_metrics = models.train_lstm(train_sessions, val_sessions, class_weights, K, label_encoder=le)
print(f"BiLSTM val macro-F1: {lstm_val_metrics['macro_f1']:.4f}")

In [ ]:
f1_matrix = pd.DataFrame(
    {
        "RF": pd.Series(rf_val_metrics["per_class_f1"]),
        "XGBoost": pd.Series(xgb_val_metrics["per_class_f1"]),
        "BiLSTM": pd.Series(lstm_val_metrics["per_class_f1"]),
    }
).T
f1_matrix.columns = [le.inverse_transform([c])[0] for c in f1_matrix.columns]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(f1_matrix, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "F1 score"})
ax.set_title("Stage 6: Per-Class Validation F1 by Model")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage6_model_f1.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 7] Adaptive Evidence Fusion

Fuses $P_A$ (RF), $P_B$ (BiLSTM), $P_C$ (XGBoost) with the pattern-mining
prior $SP_t$ and the graph consistency score $TC_t$:
$R_t = w_A P_A + w_B P_B + w_C P_C + w_S SP_t + w_T TC_t$, weights
grid-searched on validation and **frozen** before test.

> **Ordering note:** $SP_t$/$TC_t$ are Stage 8/9-10 outputs, but they only
> depend on Stage 5's training-session token sequences — not on anything
> from Stage 6 — so they are fit here (once) to give this cell everything
> the fusion search needs. Stages 8 and 9-10 below present and reuse these
> exact fitted artifacts rather than recomputing them.

In [ ]:
# --- Prerequisites for fusion (fit once here, reused by Stages 8 and 9-10 below) ---
train_token_sets = [set(t for t in rec["token_seq"] if t is not None) for rec in train_sessions]
train_session_labels = safe_label_transform(le, [rec["label"] for rec in train_sessions])

frequent_itemsets = pattern_mining.mine_fp_growth(
    train_token_sets, min_support=config.FP_GROWTH_MIN_SUPPORT,
    session_labels=train_session_labels, K=K,
)

train_events_with_ts = [
    [(t, ts) for t, ts in zip(rec["token_seq"], rec["timestamps"]) if t is not None]
    for rec in train_sessions
]
sequential_patterns = pattern_mining.mine_prefixspan(
    train_events_with_ts, min_support=config.PREFIXSPAN_MIN_SUPPORT,
    max_gap=config.PREFIXSPAN_MAX_GAP, session_labels=train_session_labels, K=K,
)

graph = attack_graph.TemporalAttackGraph()
graph.fit([[t for t in rec["token_seq"] if t is not None] for rec in train_sessions])


def compute_sp_matrix(session_records):
    return np.array([
        pattern_mining.compute_sp_score(rec, frequent_itemsets, sequential_patterns, K)
        for rec in session_records
    ])


def compute_tc_vector(session_records, graph_obj):
    return np.array([
        graph_obj.compute_tc(rec["token_seq"], rec["timestamps"]) for rec in session_records
    ])


SP_val = compute_sp_matrix(val_sessions)
TC_val = compute_tc_vector(val_sessions, graph)

val_features = build_session_features(sessions_df_val, val_sessions, feat_cols, ab_idx, bd_idx)
P_A_val, P_B_val, P_C_val = models.predict_proba_all(rf, lstm, xgb, val_features)

y_val_sessions = safe_label_transform(le, [rec["label"] for rec in val_sessions])

fusion_weights = fusion.grid_search_fusion_weights(P_A_val, P_B_val, P_C_val, SP_val, TC_val, y_val_sessions)
print(f"Best fusion weights (w_A, w_B, w_C, w_S, w_T) = {fusion_weights}")

R_t_val = fusion.fuse(P_A_val, P_B_val, P_C_val, SP_val, TC_val, fusion_weights)
val_fused_preds = np.argmax(R_t_val, axis=1)
print(f"Val macro-F1 after fusion: {f1_score(y_val_sessions, val_fused_preds, average='macro', zero_division=0):.4f}")
print("Fusion weights are now FROZEN — they will not be re-estimated on test or streaming data.")

## [STAGE 8] Frequent & Sequential Pattern Mining

FP-Growth (unordered co-occurrence) and PrefixSpan (ordered, max-gap-constrained)
patterns, mined from training sessions only. Already fit in Stage 7 (see note
there); this section presents that output.

In [ ]:
print("Top 20 frequent itemsets (FP-Growth):")
display(frequent_itemsets.sort_values("support", ascending=False).head(20)[["itemsets", "support"]])

print("\nTop 20 sequential patterns (PrefixSpan):")
for p in sequential_patterns[:20]:
    print(f"  {p['pattern']}  support={p['support']:.3f}  count={p['count']}")

print("\nExample SP_t vectors (first 5 val sessions):")
print(SP_val[:5])

## [STAGE 9-10] Temporal Attack-State Graph & Sequence Consistency

The attack-state graph (10 behavioral tokens, EMA edge weights) was fit in
Stage 7; this section visualizes it and reports $TC_t$ statistics.

In [ ]:
print("Top 10 highest-weight transitions:")
for u, v, w in graph.top_transitions(k=10):
    print(f"  {u} -> {v}: {w:.3f}")

TC_test = compute_tc_vector(test_sessions, graph)
print(f"\nTC_val stats: mean={TC_val.mean():.3f}, std={TC_val.std():.3f}")
print(f"TC_test stats: mean={TC_test.mean():.3f}, std={TC_test.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.circular_layout(graph.G)
nx.draw_networkx_nodes(graph.G, pos, node_color=[CATEGORICAL_PALETTE[4]], node_size=1400, ax=ax)
nx.draw_networkx_labels(graph.G, pos, font_size=7, ax=ax)

edges_to_draw = [(u, v) for u, v in graph.G.edges() if graph.G[u][v]["weight"] > 0.01]
weights_to_draw = [graph.G[u][v]["weight"] for u, v in edges_to_draw]
if edges_to_draw:
    nx.draw_networkx_edges(
        graph.G, pos, edgelist=edges_to_draw, width=[2 + 4 * w for w in weights_to_draw],
        edge_color=weights_to_draw, edge_cmap=plt.cm.viridis, edge_vmin=0, edge_vmax=1,
        connectionstyle="arc3,rad=0.12", ax=ax,
    )
ax.set_title("Stage 9-10: Temporal Attack-State Graph (edge width/color = transition weight)")
ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage9_graph.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 11] Adaptive Risk Decision

A logistic-regression meta-learner over $[R_t, SP_t, TC_t, G_w, 1/\Delta t]$,
fit on validation predictions/ground truth only, turns the fused evidence into
a `risk_score` and a `BENIGN`/`SUSPICIOUS`/`ATTACK` tier. Binary risk ground
truth uses each session's true (never-remapped) label, so a novel/zero-day
attack type still counts as "attack" even though the K-way classifier could
only ever call it BENIGN.

In [ ]:
def session_graph_weight(graph_obj, token_seq):
    toks = [t for t in token_seq if t is not None]
    if len(toks) <= 1:
        return 0.0
    ws = [graph_obj.G[u][v]["weight"] for u, v in zip(toks[:-1], toks[1:]) if graph_obj.G.has_edge(u, v)]
    return float(np.mean(ws)) if ws else 0.0


def mean_inter_event_seconds(timestamps):
    real_ts = sorted(t for t in timestamps if pd.notna(t))
    if len(real_ts) <= 1:
        return 0.0
    diffs = [(real_ts[i + 1] - real_ts[i]) / np.timedelta64(1, "s") for i in range(len(real_ts) - 1)]
    return float(np.mean(diffs))


test_features = build_session_features(sessions_df_test, test_sessions, feat_cols, ab_idx, bd_idx)
P_A_test, P_B_test, P_C_test = models.predict_proba_all(rf, lstm, xgb, test_features)
SP_test = compute_sp_matrix(test_sessions)
y_test_sessions = safe_label_transform(le, [rec["label"] for rec in test_sessions])

G_w_val = np.array([session_graph_weight(graph, rec["token_seq"]) for rec in val_sessions])
delta_t_train = np.array([mean_inter_event_seconds(rec["timestamps"]) for rec in train_sessions])
delta_t_mean_train = float(delta_t_train.mean()) if len(delta_t_train) else 1.0
inv_dt_val = risk.AdaptiveRiskDecider.compute_inv_dt(
    np.array([mean_inter_event_seconds(rec["timestamps"]) for rec in val_sessions]), delta_t_mean_train,
)

y_val_binary = np.array([1 if rec["label"] != "BENIGN" else 0 for rec in val_sessions])
risk_decider = risk.AdaptiveRiskDecider()
risk_decider.fit(R_t_val, SP_val, TC_val, G_w_val, inv_dt_val, y_val_binary)

R_t_test = fusion.fuse(P_A_test, P_B_test, P_C_test, SP_test, TC_test, fusion_weights)
G_w_test = np.array([session_graph_weight(graph, rec["token_seq"]) for rec in test_sessions])
inv_dt_test = risk.AdaptiveRiskDecider.compute_inv_dt(
    np.array([mean_inter_event_seconds(rec["timestamps"]) for rec in test_sessions]), delta_t_mean_train,
)

risk_score_test, alert_tier_test = risk_decider.predict(R_t_test, SP_test, TC_test, G_w_test, inv_dt_test)
print("Alert tier distribution (test):")
print(pd.Series(alert_tier_test).value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(risk_score_test, bins=40, color=CATEGORICAL_PALETTE[5])
ax.axvline(config.RISK_BENIGN_THRESH, color="black", linestyle="--", label=f"BENIGN thresh={config.RISK_BENIGN_THRESH}")
ax.axvline(config.RISK_SUSPICIOUS_THRESH, color="crimson", linestyle="--", label=f"SUSPICIOUS thresh={config.RISK_SUSPICIOUS_THRESH}")
ax.set_xlabel("Risk score")
ax.set_ylabel("Session count")
ax.set_title("Stage 11: Risk Score Distribution (Test Set)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage11_risk.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 12] Streaming Evaluation

60-second tumbling windows, 10-second stride, replayed over the test set.
RF/BiLSTM/XGBoost weights are frozen (inference only); the attack-state graph's
EMA edge weights keep adapting. This is **streaming evaluation**, not online
learning.

In [ ]:
def streaming_build_features_fn(session_records):
    return build_session_features(sessions_df_test, session_records, feat_cols, ab_idx, bd_idx)


pipeline_components = {
    "rf": rf, "lstm": lstm, "xgb": xgb, "attack_graph": graph, "risk_decider": risk_decider,
    "fusion_weights": fusion_weights, "frequent_itemsets": frequent_itemsets,
    "sequential_patterns": sequential_patterns, "K": K, "delta_t_mean_train": delta_t_mean_train,
    "build_features_fn": streaming_build_features_fn,
}

streaming_results = streaming.simulate_streaming(
    test_sessions, pipeline_components, window_size=config.STREAM_WINDOW_SIZE, stride=config.STREAM_STRIDE,
)

print(f"Streaming Macro-F1: {streaming_results['streaming_macro_f1']:.4f}")
print(f"Streaming FPR: {streaming_results['fpr']:.4f}")
print(f"Latency (ms/event): {streaming_results['latency_ms_per_event']}")
print(f"Windows processed: {streaming_results['n_windows']} | Sessions processed: {streaming_results['n_sessions_processed']}")

In [ ]:
throughput_df = pd.DataFrame(streaming_results["throughput_log"])
fig, ax = plt.subplots(figsize=(12, 6))
if len(throughput_df):
    ax.plot(throughput_df["window_start"], throughput_df["throughput_events_per_s"], color=CATEGORICAL_PALETTE[6], marker="o", markersize=3)
ax.set_xlabel("Window start time")
ax.set_ylabel("Throughput (events/s)")
ax.set_title("Stage 12: Streaming Throughput Over Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage12_streaming.png", dpi=config.FIGURE_DPI)
plt.show()

## [STAGE 13] Explainability & Evidence Chain

For 3 sample test sessions alerted as `ATTACK`, builds the full three-modality
evidence chain: TreeSHAP (RF/XGBoost), DeepSHAP with an occlusion fallback
(BiLSTM), and the concrete attack-graph path the session walked.

In [ ]:
predicted_classes_test = np.argmax(R_t_test, axis=1)
attack_test_idx = [i for i, t in enumerate(alert_tier_test) if t == "ATTACK"]
sample_idx = attack_test_idx[:3] if len(attack_test_idx) >= 3 else list(range(min(3, len(test_sessions))))

feat_names_dict = {"rf": ab_cols, "xgb": bd_cols}
background_tokens = np.array([r["token_ids"] for r in train_sessions[:config.SHAP_SAMPLE_SIZE]], dtype=np.int32)

evidence_reports = []
for i in sample_idx:
    rec = test_sessions[i]
    session_obj = {
        "session_id": rec["session_id"],
        "risk_score": float(risk_score_test[i]),
        "alert_tier": alert_tier_test[i],
        "predicted_class": int(predicted_classes_test[i]),
        "X_AB": test_features["X_AB"][i],
        "X_BD": test_features["X_BD"][i],
        "token_ids": test_features["token_ids"][i],
        "token_seq": rec["token_seq"],
        "background_tokens": background_tokens,
    }
    report = explainability.generate_evidence_chain(session_obj, rf, xgb, lstm, graph, feat_names_dict, top_k=5)
    evidence_reports.append(report)

    print(f"\n=== Evidence Chain: {report['session_id']} ===")
    print(f"Risk: {report['risk_score']:.3f} ({report['alert_tier']})")
    print(f"Attack family: {report['attack_family']} | Kill-chain stage: {report['kill_chain_stage']}")
    print(f"Top RF features: {report['top_features_rf']}")
    print(f"Top XGBoost features: {report['top_features_xgb']}")
    print(f"LSTM token attributions: {report['lstm_token_attributions']}")
    print(f"Graph path: {report['graph_path']}")
    print(f"Novel edges: {report['novel_edges']}")
    print(f"Recommended action: {report['recommended_action']}")
    print(f"Analyst summary: {report['analyst_summary']}")

In [ ]:
import shap

rng = np.random.default_rng(config.SEED)
sample_n = min(100, len(X_val_AB))
shap_sample_idx = rng.choice(len(X_val_AB), size=sample_n, replace=False)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_val_AB[shap_sample_idx])

sv_for_plot = shap_values[0] if isinstance(shap_values, list) else np.asarray(shap_values)[..., 0]

plt.figure(figsize=(10, 8))
shap.summary_plot(sv_for_plot, X_val_AB[shap_sample_idx], feature_names=ab_cols, show=False)
plt.title("Stage 13: SHAP Summary (Random Forest, class 0)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage13_shap.png", dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

## [RESULTS] Final Metrics

Per-class precision/recall/F1 for all classes, overall macro-F1/accuracy,
streaming macro-F1/FPR/latency, and a comparison of each standalone detector
against the full fused system.

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_test_sessions, predicted_classes_test, labels=list(range(K)), zero_division=0,
)
results_table = pd.DataFrame({
    "class": [le.inverse_transform([i])[0] for i in range(K)],
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
})

overall_macro_f1 = f1_score(y_test_sessions, predicted_classes_test, average="macro", zero_division=0)
overall_accuracy = accuracy_score(y_test_sessions, predicted_classes_test)

print(results_table.to_string(index=False))
print(f"\nOverall Macro-F1 (test, fused system): {overall_macro_f1:.4f}")
print(f"Overall Accuracy (test, fused system): {overall_accuracy:.4f}")
print(f"Streaming FPR: {streaming_results['fpr']:.4f}")
print(f"Streaming Macro-F1: {streaming_results['streaming_macro_f1']:.4f}")
print(f"Mean latency: {streaming_results['latency_ms_per_event']['mean']:.3f} ms/event")

In [ ]:
rf_only_preds = np.argmax(P_A_test, axis=1)
xgb_only_preds = np.argmax(P_C_test, axis=1)
lstm_only_preds = np.argmax(P_B_test, axis=1)

comparison = pd.DataFrame({
    "system": ["RF-only", "XGBoost-only", "BiLSTM-only", "Full Fused System"],
    "test_macro_f1": [
        f1_score(y_test_sessions, rf_only_preds, average="macro", zero_division=0),
        f1_score(y_test_sessions, xgb_only_preds, average="macro", zero_division=0),
        f1_score(y_test_sessions, lstm_only_preds, average="macro", zero_division=0),
        f1_score(y_test_sessions, predicted_classes_test, average="macro", zero_division=0),
    ],
})
print("Model comparison (test macro-F1):")
print(comparison.to_string(index=False))

results_table.to_csv(RESULTS_DIR / "results_table.csv", index=False)
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'results_table.csv'}")
print(f"Saved: {RESULTS_DIR / 'model_comparison.csv'}")